# 07 - exploration

three separate analyses here. first we do a visual exploration of the weather-demand relationship - lag correlations, scatter plots, time series overlays, seasonal breakdowns. then we run the formal statistical tests that go in the paper (mann-whitney u, ks, shapiro-wilk, ljung-box). finally we generate the two ieee paper figures - the cleaning boxplots and the weather outlier flag counts.

## weather-demand visual exploration

load the joined demand-weather parquet and aggregate up to city-wide hourly. everything below works on that aggregated view.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', font_scale=1.1)

ROOT     = Path('/Users/Xavier/Desktop/university/YEAR4/datascienceNO3')
PATH     = ROOT / 'exports' / 'demand_weather_joined_2023_2024.parquet'
OUT_DIR  = ROOT / 'outputs' / 'exploration'
OUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET       = 'total_trips'
WEATHER_COLS = ['temperature_2m', 'precipitation', 'snowfall', 'windspeed_10m', 'weathercode']
WEATHER_LABELS = {
    'temperature_2m': 'Temperature (C)',
    'precipitation':  'Precipitation (mm)',
    'snowfall':       'Snowfall (mm)',
    'windspeed_10m':  'Wind Speed (km/h)',
    'weathercode':    'Weather Code',
}

print('loading data...')
df = pd.read_parquet(PATH)
print(f'  {len(df):,} rows  |  columns: {list(df.columns)}')

df['hour_slot'] = pd.to_datetime(df['hour_slot'])

for col in WEATHER_COLS:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

print('aggregating to city-wide hourly...')
hourly = df.groupby('hour_slot').agg(
    {TARGET: 'sum', **{col: 'mean' for col in WEATHER_COLS}}
).dropna()
print(f'  {len(hourly):,} hourly rows  |  demand range: {hourly[TARGET].min():.0f}-{hourly[TARGET].max():.0f} trips/hr')

## lag correlation heatmap

for each weather variable we compute the pearson r between city-wide demand and lagged versions of that variable (0 to 12 hours). shows whether weather has any leading effect on demand.

In [ ]:
print('plot 1: lag correlation heatmap...')
lag_results = {}
for col in WEATHER_COLS:
    corrs = []
    for lag in range(0, 13):
        correlation = hourly[TARGET].corr(hourly[col].shift(lag))
        corrs.append(round(correlation, 3))
    lag_results[WEATHER_LABELS[col]] = corrs

lag_df = pd.DataFrame(lag_results, index=[f'{i}h' for i in range(13)])

plt.figure(figsize=(13, 5))
sns.heatmap(lag_df.T, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.4, cbar_kws={'label': 'Pearson r'})
plt.title('Correlation: City-Wide Hourly Demand vs Lagged Weather (0-12h)',
          fontsize=13, pad=12)
plt.xlabel('Time Lag'); plt.ylabel('Weather Feature')
plt.tight_layout()
plt.savefig(OUT_DIR / 'lag_correlation_heatmap.png', dpi=150)
plt.show()
print(f'  saved -> {OUT_DIR}/lag_correlation_heatmap.png')

## demand vs weather scatter plots

we bin each weather variable into 30 quantile buckets and plot mean demand per bucket with a 1-std band. easier to read than a raw scatter when you have 17k+ hourly points.

In [ ]:
print('plot 2: scatter plots demand vs weather...')
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, col in enumerate(WEATHER_COLS):
    ax = axes[i]
    s  = hourly[[col, TARGET]].dropna().sort_values(col)
    try:
        bins = pd.qcut(s[col], q=30, duplicates='drop')
        grp  = s.groupby(bins, observed=True)[TARGET].agg(['mean', 'std'])
        mids = [iv.mid for iv in grp.index.categories]
        step = max(1, len(grp) // 6)
        ax.fill_between(range(len(grp)),
                        grp['mean'] - grp['std'],
                        grp['mean'] + grp['std'],
                        alpha=0.2, color='steelblue')
        ax.plot(range(len(grp)), grp['mean'], color='steelblue', linewidth=2)
        ax.set_xticks(range(0, len(grp), step))
        ax.set_xticklabels([f'{mids[j]:.1f}' for j in range(0, len(grp), step)],
                           rotation=30, fontsize=8)
    except Exception:
        ax.scatter(s[col], s[TARGET], alpha=0.1, s=2, color='steelblue')
    r = hourly[TARGET].corr(hourly[col])
    ax.set_xlabel(WEATHER_LABELS[col], fontsize=9)
    ax.set_ylabel('Total Trips (city-wide/hr)', fontsize=9)
    ax.set_title(f"{WEATHER_LABELS[col]}  (r={r:+.3f})", fontsize=9)
    ax.grid(True, alpha=0.3)

axes[-1].axis('off')
plt.suptitle('City-Wide Demand vs Weather (quantile bins, mean +/- 1 std)', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'demand_vs_weather_scatter.png', dpi=150)
plt.show()

## time series overlay

two weeks of january 2024 with demand, temperature and precipitation stacked. useful sanity check that the data looks sensible.

In [ ]:
print('plot 3: time series overlay...')
sample = hourly.loc['2024-01-01':'2024-01-14']

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
axes[0].plot(sample.index, sample[TARGET], color='steelblue', linewidth=1)
axes[0].set_ylabel('Total Trips/hr')
axes[0].set_title('City-Wide Demand - 2 weeks (Jan 2024)')
axes[1].plot(sample.index, sample['temperature_2m'], color='darkorange', linewidth=1)
axes[1].set_ylabel('Temp (C)')
axes[2].bar(sample.index, sample['precipitation'], color='royalblue', width=0.03, alpha=0.8)
axes[2].plot(sample.index, sample['snowfall'], color='grey', linewidth=1, linestyle='--', label='Snowfall')
axes[2].set_ylabel('Precip / Snow (mm)')
axes[2].legend(fontsize=8)
axes[2].set_xlabel('Date')
plt.tight_layout()
plt.savefig(OUT_DIR / 'timeseries_demand_weather.png', dpi=150)
plt.show()

## demand by precipitation category

we cut precipitation into dry / light / moderate / heavy bins and look at how the demand distribution changes. also plots dry vs rainy mean hourly curves side by side.

In [ ]:
print('plot 4: demand by precipitation category...')
hourly['rain_cat'] = pd.cut(
    hourly['precipitation'],
    bins=[-0.001, 0.0, 1.0, 5.0, 1000],
    labels=['Dry', 'Light (0-1mm)', 'Moderate (1-5mm)', 'Heavy (>5mm)']
)
hourly['hour_of_day'] = hourly.index.hour

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
hourly.boxplot(column=TARGET, by='rain_cat', ax=axes[0],
               boxprops=dict(color='steelblue'),
               medianprops=dict(color='darkorange', linewidth=2),
               flierprops=dict(marker='.', alpha=0.2))
axes[0].set_title('Demand Distribution by Precipitation Category')
axes[0].set_xlabel(''); axes[0].set_ylabel('Total Trips/hr')
plt.sca(axes[0]); plt.xticks(rotation=20)

dry  = hourly[hourly['rain_cat'] == 'Dry'].groupby('hour_of_day')[TARGET].mean()
rain = hourly[hourly['rain_cat'] != 'Dry'].groupby('hour_of_day')[TARGET].mean()
axes[1].plot(dry.index,  dry.values,  label='Dry',   color='steelblue',  linewidth=2)
axes[1].plot(rain.index, rain.values, label='Rainy',  color='darkorange', linewidth=2)
axes[1].set_xlabel('Hour of Day'); axes[1].set_ylabel('Mean Total Trips/hr')
axes[1].set_title('Mean Hourly Demand: Dry vs Rainy')
axes[1].legend(); axes[1].set_xticks(range(0, 24, 2))
plt.suptitle('')
plt.tight_layout()
plt.savefig(OUT_DIR / 'demand_by_rain_category.png', dpi=150)
plt.show()

## seasonal analysis

weather-demand correlation broken down by season (winter/spring/summer/autumn), plus mean demand by season and rain category.

In [ ]:
print('plot 5: seasonal analysis...')
hourly['month']  = hourly.index.month
hourly['season'] = hourly['month'].map({
    12:'Winter', 1:'Winter', 2:'Winter',
    3:'Spring',  4:'Spring', 5:'Spring',
    6:'Summer',  7:'Summer', 8:'Summer',
    9:'Autumn',  10:'Autumn',11:'Autumn'
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
season_order = ['Winter', 'Spring', 'Summer', 'Autumn']
season_corrs = {}
for season in season_order:
    s = hourly[hourly['season'] == season]
    season_corrs[season] = {
        WEATHER_LABELS[col]: round(s[TARGET].corr(s[col]), 3)
        for col in WEATHER_COLS
    }
sc_df = pd.DataFrame(season_corrs).T
sns.heatmap(sc_df, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            ax=axes[0], linewidths=0.4, cbar_kws={'label': 'r'})
axes[0].set_title('Weather-Demand Correlation by Season')

season_rain = hourly.groupby(['season', 'rain_cat'], observed=True)[TARGET].mean().unstack()
season_rain = season_rain.loc[season_order]
season_rain.plot(kind='bar', ax=axes[1], colormap='Blues', edgecolor='white')
axes[1].set_title('Mean Demand by Season and Rain Category')
axes[1].set_ylabel('Mean Total Trips/hr')
axes[1].legend(title='Rain', fontsize=8)
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(OUT_DIR / 'seasonal_weather_effect.png', dpi=150)
plt.show()

print('\nsummary: weather-demand correlations (city-wide hourly)')
for col in WEATHER_COLS:
    r = hourly[TARGET].corr(hourly[col])
    print(f'  {WEATHER_LABELS[col]:<30}  r = {r:+.4f}')

## statistical tests

now the formal stuff. mann-whitney u tests on hourly demand under different weather conditions (rain, wind, snow vs calm). then ks + shapiro-wilk on raw yellow trip data to justify the 3x iqr fence. then ljung-box on the aggregate demand time series to confirm strong autocorrelation.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.ticker as mticker
from scipy import stats
from scipy.stats import mannwhitneyu, ks_2samp, shapiro
from statsmodels.stats.diagnostic import acorr_ljungbox

BASE   = Path('/Users/Xavier/Desktop/university/YEAR4/datascienceNO3')
JFILE  = BASE / 'exports/demand_weather_joined_2023_2024.parquet'
RAW_Y  = BASE / 'raw/yellow'
OUT    = BASE / 'outputs/weather_analysis'
OUT.mkdir(parents=True, exist_ok=True)

RAIN_MM  = 0.1
SNOW_CM  = 0.1
WIND_KMH = 20.0

SEASONS = {
    'Winter': [12, 1, 2],
    'Spring': [3, 4, 5],
    'Summer': [6, 7, 8],
    'Fall':   [9, 10, 11],
}
DAYS   = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
COLORS = {'None': '#2196F3', 'Rain': '#4CAF50', 'Wind': '#FF9800', 'Snow': '#9C27B0'}
ALPHA  = {'None': 1.0, 'Rain': 0.85, 'Wind': 0.85, 'Snow': 0.85}

print('loading demand-weather data...')
df2 = pd.read_parquet(JFILE)

df2['hour']     = df2['hour_slot'].dt.hour
df2['month']    = df2['hour_slot'].dt.month
df2['dow']      = df2['hour_slot'].dt.dayofweek
df2['dow_name'] = df2['dow'].map(dict(enumerate(DAYS)))

def month_to_season(m):
    for s, months in SEASONS.items():
        if m in months:
            return s
    return 'Unknown'

df2['season'] = df2['month'].map(month_to_season)

df2['is_rain'] = df2['precipitation'] > RAIN_MM
df2['is_snow'] = df2['snowfall']      > SNOW_CM
df2['is_wind'] = df2['windspeed_10m'] > WIND_KMH
df2['is_none'] = ~df2['is_rain'] & ~df2['is_snow'] & ~df2['is_wind']

grp_cols = ['hour_slot', 'season', 'dow', 'dow_name', 'hour',
            'is_rain', 'is_snow', 'is_wind', 'is_none']
hourly2 = (
    df2.groupby(grp_cols, observed=True)['total_trips']
    .sum()
    .reset_index()
)

## demand cycle plots

for each season we plot mean hourly demand per day of week, with separate lines for calm / rain / wind / snow conditions. shows how weather suppresses or shifts the typical daily demand curve.

In [ ]:
# build mean daily demand curves per season x dow x weather condition
records = []
for season in SEASONS:
    for dow_i, dow_name in enumerate(DAYS):
        mask_base = (hourly2['season'] == season) & (hourly2['dow'] == dow_i)
        for cond, flag in [('None','is_none'),('Rain','is_rain'),
                           ('Wind','is_wind'),('Snow','is_snow')]:
            sub = hourly2[mask_base & hourly2[flag]]
            if sub.empty:
                curve = pd.Series(np.nan, index=range(24))
            else:
                curve = sub.groupby('hour')['total_trips'].mean().reindex(range(24))
            for h, v in curve.items():
                records.append(dict(season=season, dow=dow_i, dow_name=dow_name,
                                    cond=cond, hour=h, mean_trips=v))

curves = pd.DataFrame(records)

print('plotting demand cycle graphs...')
for season in SEASONS:
    fig, axes = plt.subplots(2, 4, figsize=(22, 10), sharey=False)
    axes = axes.flatten()
    fig.suptitle(f'Hourly Demand by Weather Condition - {season}',
                 fontsize=15, fontweight='bold', y=1.01)

    for di, dow_name in enumerate(DAYS):
        ax  = axes[di]
        sub = curves[(curves['season'] == season) & (curves['dow_name'] == dow_name)]
        for cond in ['None', 'Rain', 'Wind', 'Snow']:
            row = sub[sub['cond'] == cond].sort_values('hour')
            ax.plot(row['hour'], row['mean_trips'],
                    label=cond, color=COLORS[cond], linewidth=2.2, alpha=ALPHA[cond],
                    linestyle=('solid' if cond == 'None' else 'dashed'))
        ax.set_title(dow_name, fontsize=11, fontweight='bold')
        ax.set_xlabel('Hour of day', fontsize=9)
        ax.set_ylabel('Mean trips', fontsize=9)
        ax.set_xticks(range(0, 24, 3))
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=7, loc='upper left')

    axes[-1].axis('off')
    fig.tight_layout()
    fpath = OUT / f'demand_cycle_{season.lower()}.png'
    fig.savefig(fpath, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'  saved {fpath.name}')

## percentage difference table

how much does rain/wind/snow change the total daily demand compared to calm conditions, broken down by season and day of week.

In [ ]:
print('building percentage-difference table...')
daily = (
    curves.groupby(['season','dow_name','cond'])['mean_trips']
    .sum()
    .reset_index()
    .rename(columns={'mean_trips': 'daily_total'})
)
piv = daily.pivot_table(index=['season','dow_name'],
                        columns='cond', values='daily_total').reset_index()
piv.columns.name = None

for cond in ['Rain', 'Wind', 'Snow']:
    diff         = piv[cond] - piv['None']
    piv[f'pct_{cond}'] = (diff / piv['None'] * 100).round(2)

season_order = list(SEASONS.keys())
piv['season_ord'] = piv['season'].map({s: i for i, s in enumerate(season_order)})
piv['dow_ord']    = piv['dow_name'].map({d: i for i, d in enumerate(DAYS)})
piv = piv.sort_values(['season_ord','dow_ord']).drop(columns=['season_ord','dow_ord'])

pct_cols  = [c for c in piv.columns if c.startswith('pct_')]
table_out = piv[['season','dow_name'] + pct_cols].copy()
table_out.columns = ['Season','Day','Delta% Rain vs None','Delta% Wind vs None','Delta% Snow vs None']

print(table_out.to_string(index=False))
table_out.to_csv(OUT / 'pct_diff_table.csv', index=False)

## mann-whitney u tests

for every season x day of week combination, we test whether the hourly demand distribution under each weather condition (rain, wind, snow) is significantly different from calm. effect size r is included.

In [ ]:
print('running mann-whitney u tests...')
stat_rows = []
for season in SEASONS:
    for dow_i, dow_name in enumerate(DAYS):
        mask_base  = (hourly2['season'] == season) & (hourly2['dow'] == dow_i)
        none_vals  = hourly2[mask_base & hourly2['is_none']]['total_trips'].values
        for cond, flag in [('Rain','is_rain'),('Wind','is_wind'),('Snow','is_snow')]:
            cond_vals = hourly2[mask_base & hourly2[flag]]['total_trips'].values
            if len(none_vals) < 5 or len(cond_vals) < 5:
                p = np.nan; stat = np.nan
            else:
                stat, p = mannwhitneyu(none_vals, cond_vals, alternative='two-sided')
            n1, n2 = len(none_vals), len(cond_vals)
            if not np.isnan(p) and (n1+n2) > 0:
                z     = stats.norm.ppf(1 - p/2) * np.sign(np.median(cond_vals) - np.median(none_vals))
                r_eff = z / np.sqrt(n1 + n2)
            else:
                r_eff = np.nan
            if not np.isnan(p):
                sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
            else:
                sig = 'NA'
            stat_rows.append(dict(
                Season=season, Day=dow_name, Condition=cond,
                N_None=n1, N_Cond=n2,
                U_stat=round(stat, 1) if not np.isnan(stat) else np.nan,
                p_value=round(p, 6) if not np.isnan(p) else np.nan,
                sig=sig,
                r_effect=round(r_eff, 3) if not np.isnan(r_eff) else np.nan,
            ))

stat_df = pd.DataFrame(stat_rows)
print(stat_df.to_string(index=False))
stat_df.to_csv(OUT / 'mannwhitney_tests.csv', index=False)

sig_summary = stat_df.groupby(['Condition','sig']).size().unstack(fill_value=0)
print('\nsignificance summary (counts across all season x day combos):')
print(sig_summary)

## ks + shapiro-wilk tests

we load raw yellow trip data (2023-2024) and test whether trip distance and fare are normally distributed before and after 3x iqr cleaning. the answer is no - they're heavily right-skewed - which is why iqr outlier removal is appropriate here rather than z-score methods.

In [ ]:
print('loading raw yellow taxi data for ks/shapiro-wilk...')
all_trips = []
for yr in [2023, 2024]:
    for mo in range(1, 13):
        fp = RAW_Y / f'yellow_tripdata_{yr}-{mo:02d}.parquet'
        if fp.exists():
            chunk = pd.read_parquet(fp, columns=['trip_distance', 'fare_amount'])
            all_trips.append(chunk)

raw = pd.concat(all_trips, ignore_index=True)
print(f'  loaded {len(raw):,} raw trips (2023-2024)')

def iqr_fence(s, k=3):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr    = q3 - q1
    lo     = q1 - k * iqr
    hi     = q3 + k * iqr
    return s[(s >= lo) & (s <= hi) & (s > 0)]

dist_raw   = raw['trip_distance'].dropna()
fare_raw   = raw['fare_amount'].dropna()
dist_clean = iqr_fence(dist_raw)
fare_clean = iqr_fence(fare_raw)

SAMPLE_N = 5000
rng = np.random.default_rng(42)

for label, series in [('trip_distance (raw)', dist_raw),
                       ('trip_distance (3xiqr cleaned)', dist_clean),
                       ('fare_amount (raw)', fare_raw),
                       ('fare_amount (3xiqr cleaned)', fare_clean)]:
    samp          = rng.choice(series.values, size=min(SAMPLE_N, len(series)), replace=False)
    sw_stat, sw_p = shapiro(samp)
    mu, sigma     = samp.mean(), samp.std()
    ks_stat, ks_p = ks_2samp(samp, rng.normal(mu, sigma, size=len(samp)))
    sw_result = 'reject normality' if sw_p < 0.05 else 'fail to reject'
    ks_result = 'non-normal' if ks_p < 0.05 else 'approx normal'
    print(f'\n  {label}')
    print(f'    shapiro-wilk : W={sw_stat:.5f}, p={sw_p:.2e}  ({sw_result})')
    print(f'    ks vs normal : D={ks_stat:.5f}, p={ks_p:.2e}  ({ks_result})')

for label, s in [('trip_distance clean', dist_clean), ('fare_amount clean', fare_clean)]:
    print(f'\n  {label}: skewness={s.skew():.3f}, excess kurtosis={s.kurtosis():.3f}')

In [ ]:
import matplotlib
matplotlib.use('inline')
import importlib
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, (label, s, color) in zip(axes.flatten(), [
        ('Trip Distance (raw)', dist_raw, '#E53935'),
        ('Trip Distance (3xIQR)', dist_clean, '#1E88E5'),
        ('Fare Amount (raw)', fare_raw, '#43A047'),
        ('Fare Amount (3xIQR)', fare_clean, '#FB8C00'),
    ]):
    samp      = rng.choice(s.values, size=min(50_000, len(s)), replace=False)
    mu, sigma = samp.mean(), samp.std()
    x         = np.linspace(samp.min(), samp.max(), 400)
    ax.hist(samp, bins=100, color=color, alpha=0.7, density=True, label='Empirical')
    ax.plot(x, stats.norm.pdf(x, mu, sigma), 'k--', lw=1.8, label='Normal fit')
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Value'); ax.set_ylabel('Density')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
fig.suptitle('Distribution vs Normal Fit - justifying 3xIQR fence', fontsize=12, fontweight='bold')
fig.tight_layout()
fig.savefig(OUT / 'ks_shapiro_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## ljung-box test

tests whether the aggregate hourly demand time series has any significant autocorrelation at lags 1, 2, 3, 4, 12, 24, 48 and 168 hours. the strong spikes at 24h and 168h confirm the daily and weekly seasonality that justifies our lag features.

In [ ]:
print('running ljung-box test on aggregate hourly demand...')
ts = (
    df2.groupby('hour_slot')['total_trips']
    .sum()
    .sort_index()
    .asfreq('h')
    .fillna(0)
)
print(f'  time series: {len(ts):,} hourly observations  ({ts.index[0].date()} to {ts.index[-1].date()})')

lags_to_test = [1, 2, 3, 4, 12, 24, 48, 168]
lb = acorr_ljungbox(ts, lags=max(lags_to_test), return_df=True)
lb_subset = lb.loc[[l for l in lags_to_test if l <= len(lb)]]

print(f'\n{"Lag":>5}  {"Q-stat":>12}  {"p-value":>12}  Sig')
print('-'*45)
for lag, row in lb_subset.iterrows():
    p   = row['lb_pvalue']
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    print(f'{lag:>5}  {row["lb_stat"]:>12.2f}  {p:>12.2e}  {sig}')

lb_subset.index.name = 'lag'
lb_subset.to_csv(OUT / 'ljungbox_results.csv')

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9))

plot_acf(ts, lags=200, ax=ax1, alpha=0.05, zero=False)
ax1.set_title('ACF - Aggregate Hourly NYC Taxi Demand', fontweight='bold')
ax1.set_xlabel('Lag (hours)'); ax1.set_ylabel('Autocorrelation')
ax1.grid(True, alpha=0.3)

lb_full = acorr_ljungbox(ts, lags=200, return_df=True)
ax2.semilogy(lb_full.index, lb_full['lb_pvalue'], color='#E53935', linewidth=1.5)
ax2.axhline(0.05,  color='k',    linestyle='--', lw=1, label='p = 0.05')
ax2.axhline(0.001, color='grey', linestyle=':',  lw=1, label='p = 0.001')
ax2.set_title('Ljung-Box p-values by Lag', fontweight='bold')
ax2.set_xlabel('Lag (hours)'); ax2.set_ylabel('p-value (log scale)')
ax2.legend(); ax2.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(OUT / 'ljungbox_acf.png', dpi=150, bbox_inches='tight')
plt.show()

lag1_acf = ts.autocorr(lag=1)
print(f'\n  lag-1 pearson autocorrelation: r = {lag1_acf:.4f}')
print(f'  ljung-box Q(1) = {lb.loc[1,"lb_stat"]:.2f}, p = {lb.loc[1,"lb_pvalue"]:.2e}')

## ieee paper figures

these are the two figures that go in the paper. figure 1 is boxplots of trip-level distributions (distance, fare, speed) for yellow vs green cabs after cleaning. figure 2 is a bar chart of iqr-flagged observations per weather variable.

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter

matplotlib.rcParams.update({
    'font.family':       'sans-serif',
    'font.sans-serif':   ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size':         7,
    'axes.titlesize':    7,
    'axes.labelsize':    7,
    'xtick.labelsize':   6.5,
    'ytick.labelsize':   6.5,
    'legend.fontsize':   6.5,
    'axes.linewidth':    0.6,
    'xtick.major.width': 0.6,
    'ytick.major.width': 0.6,
    'xtick.major.size':  2.5,
    'ytick.major.size':  2.5,
    'pdf.fonttype':      42,
    'ps.fonttype':       42,
})

DATA_DIR = '/Users/Xavier/Desktop/university/YEAR4/datascienceNO3/data'

GOLD  = '#C9A84C'
GREEN = '#4E8B5F'

In [ ]:
print('loading parquet files...')
SAMPLE = 120_000

rng2 = np.random.default_rng(42)

def load_sample(path, n, rng):
    df_s  = pd.read_parquet(path, columns=[
        'trip_distance_mi', 'fare', 'speed_mph',
        'is_dist_outlier', 'is_fare_outlier', 'is_speed_outlier'
    ])
    idx = rng.choice(len(df_s), size=min(n, len(df_s)), replace=False)
    return df_s.iloc[idx].reset_index(drop=True)

y = load_sample(f'{DATA_DIR}/clean_yellow.parquet', SAMPLE, rng2)
g = load_sample(f'{DATA_DIR}/clean_green.parquet',  SAMPLE, rng2)
print(f'  yellow sample: {len(y):,}  |  green sample: {len(g):,}')

def clean_var(df_in, col, flag_col):
    mask = (df_in[flag_col] == False) & df_in[col].notna() & np.isfinite(df_in[col]) & (df_in[col] >= 0)
    return df_in.loc[mask, col].values

data = {
    'dist': {
        'yellow': clean_var(y, 'trip_distance_mi', 'is_dist_outlier'),
        'green':  clean_var(g, 'trip_distance_mi', 'is_dist_outlier'),
        'whis':   1.5,
        'ylabel': 'Trip Distance (mi)',
        'xlabel': 'Trip Distance',
    },
    'fare': {
        'yellow': clean_var(y, 'fare', 'is_fare_outlier'),
        'green':  clean_var(g, 'fare', 'is_fare_outlier'),
        'whis':   3.0,
        'ylabel': 'Fare Amount ($)',
        'xlabel': 'Fare Amount',
    },
    'speed': {
        'yellow': clean_var(y, 'speed_mph', 'is_speed_outlier'),
        'green':  clean_var(g, 'speed_mph', 'is_speed_outlier'),
        'whis':   3.0,
        'ylabel': 'Average Speed (mph)',
        'xlabel': 'Average Speed',
    },
}

In [ ]:
# figure 1 - cleaning boxplots
fig, axes = plt.subplots(1, 3, figsize=(7.0, 2.6))
fig.subplots_adjust(wspace=0.38, left=0.08, right=0.97, top=0.88, bottom=0.16)

def make_bp_props(color):
    lw = 0.8
    return dict(
        boxprops     = dict(color=color, linewidth=lw),
        whiskerprops = dict(color=color, linewidth=lw),
        capprops     = dict(color=color, linewidth=lw),
        medianprops  = dict(color='white', linewidth=1.2, solid_capstyle='butt'),
        flierprops   = dict(marker='o', markersize=1.2, alpha=0.18,
                            markerfacecolor=color, markeredgewidth=0, linestyle='none'),
    )

for ax, (key, info) in zip(axes, data.items()):
    yd   = info['yellow']
    gd   = info['green']
    whis = info['whis']

    bp_y = ax.boxplot(yd, positions=[1], widths=0.52, patch_artist=True,
                      whis=whis, showfliers=True, **make_bp_props(GOLD))
    bp_g = ax.boxplot(gd, positions=[2], widths=0.52, patch_artist=True,
                      whis=whis, showfliers=True, **make_bp_props(GREEN))

    bp_y['boxes'][0].set_facecolor(GOLD);  bp_y['boxes'][0].set_alpha(0.82)
    bp_g['boxes'][0].set_facecolor(GREEN); bp_g['boxes'][0].set_alpha(0.82)

    ax.set_xlim(0.3, 2.7)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['Yellow', 'Green'], fontsize=6.5)
    ax.set_ylabel(info['ylabel'], labelpad=3)
    ax.set_xlabel(info['xlabel'], labelpad=4, fontsize=7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='x', length=0)
    ax.yaxis.set_tick_params(labelsize=6)
    ax.grid(axis='y', linewidth=0.4, color='#dddddd', zorder=0)
    ax.set_axisbelow(True)

fig.suptitle('Trip-level distributions by cab type (post-cleaning)', fontsize=7.5, y=0.97)

for fmt in ['pdf', 'png']:
    path = f'{DATA_DIR}/cleaning_boxplot.{fmt}'
    fig.savefig(path, format=fmt, dpi=300, bbox_inches='tight',
                facecolor='white', edgecolor='none')
    print(f'saved: {path}')
plt.show()

In [ ]:
# figure 2 - weather outlier flags
outlier_csv = '/Users/Xavier/Desktop/university/YEAR4/datascienceNO3/weather/outlier_report.csv'
weather_counts = (
    pd.read_csv(outlier_csv)
    .groupby('column', as_index=False)
    .size()
    .rename(columns={'column': 'variable', 'size': 'count'})
    .sort_values('count', ascending=True)
)

BLUE   = '#5B8DB8'
ORANGE = '#D4865A'

colors2 = [ORANGE if v == 'windspeed_10m' else BLUE
          for v in weather_counts['variable']]

label_map = {
    'precipitation': 'Precipitation',
    'snow_depth':    'Snow Depth',
    'snowfall':      'Snowfall',
    'windspeed_10m': 'Wind Speed (10 m)',
}
labels2 = [label_map[v] for v in weather_counts['variable']]

fig2, ax2 = plt.subplots(figsize=(3.5, 2.2))
fig2.subplots_adjust(left=0.28, right=0.97, top=0.88, bottom=0.18)

ax2.barh(labels2, weather_counts['count'], color=colors2, height=0.55, edgecolor='none')

def thousands_fmt(x, _):
    return '0' if x == 0 else f'{x/1000:.0f}k'

ax2.xaxis.set_major_formatter(FuncFormatter(thousands_fmt))
ax2.set_xlabel('Flagged observations', labelpad=4)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.spines['left'].set_visible(False)
ax2.tick_params(axis='y', length=0)
ax2.grid(axis='x', linewidth=0.4, color='#dddddd', zorder=0)
ax2.set_axisbelow(True)

precip_count = weather_counts.loc[weather_counts['variable'] == 'precipitation', 'count'].values[0]
precip_bar_y = len(labels2) - 1 - list(weather_counts['variable'][::-1]).index('precipitation')
ax2.text(
    precip_count * 0.99, precip_bar_y,
    'IQR degeneracy:\nQ1=Q3=0 (~85% dry hours)',
    ha='right', va='center', fontsize=5.5, color='white',
    fontweight='bold', linespacing=1.35,
)

ax2.set_title('IQR-flagged weather observations by variable', fontsize=7, pad=6)

patch_blue   = mpatches.Patch(facecolor=BLUE,   edgecolor='none', label='Precip / Snow')
patch_orange = mpatches.Patch(facecolor=ORANGE, edgecolor='none', label='Wind')
ax2.legend(handles=[patch_blue, patch_orange], fontsize=6,
           frameon=False, loc='lower right',
           bbox_to_anchor=(1.0, 0.01), ncol=2, handlelength=1.0, handletextpad=0.4)

for fmt in ['pdf', 'png']:
    path = f'{DATA_DIR}/weather_outlier_flags.{fmt}'
    fig2.savefig(path, format=fmt, dpi=300, bbox_inches='tight',
                 facecolor='white', edgecolor='none')
    print(f'saved: {path}')
plt.show()
print('done')